# sphinx_basic_ng

sphinx-basic-ng 1.0.0b2 is a modernised skeleton for Sphinx HTML themes: the base Furo is built on (python/furo
requires it). It registers a `basic-ng` theme whose templates are building blocks ("components") other themes extend.
These cells build with it directly, check its components, its registration and Furo's use of it.

In [ ]:
# Example 1: a site builds with the basic-ng theme itself (a page skeleton: themes that extend it fill the content).
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'B'\nhtml_theme = 'basic-ng'\n")
(src / "index.rst").write_text("B\n=\n\nHello basic-ng.\n")
rc = build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")])
print("rc", rc)
html = (src / "_out" / "index.html").read_text()
assert rc == 0 and "<title>B</title>" in html and "_static/skeleton.css" in html  # a skeleton: themes fill the body

In [ ]:
# Example 2: the theme ships its building-block templates ("components") for themes to include.
import pathlib
import sphinx_basic_ng

theme = pathlib.Path(sphinx_basic_ng.__file__).parent / "theme" / "basic-ng"
components = sorted(p.name for p in (theme / "components").iterdir())
print(components)
assert (theme / "theme.conf").exists() or (theme / "theme.toml").exists()
assert {"search-input.html", "view-this-page.html", "edit-this-page.html"} <= set(components)

In [ ]:
# Example 3: the theme is registered through Sphinx's entry point, and its setup returns parallel-safe metadata.
import importlib.metadata as md
import sphinx_basic_ng

entry_points = {ep.name: ep.value for ep in md.entry_points(group="sphinx.html_themes")}
print(entry_points.get("basic-ng"))
assert entry_points.get("basic-ng") == "sphinx_basic_ng"
assert md.version("sphinx-basic-ng") == "1.0.0b2"
assert callable(sphinx_basic_ng.setup)

In [ ]:
# Example 4: Furo inherits from it: Furo's theme names basic-ng as its parent.
import pathlib
import furo

conf = pathlib.Path(furo.__file__).parent / "theme" / "furo" / "theme.conf"
text = conf.read_text()
print(text[:200])
assert "inherit = basic-ng" in text

In [ ]:
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
theme = src / "_themes" / "mine"
theme.mkdir(parents=True)
(theme / "theme.conf").write_text("[theme]\ninherit = basic-ng\nstylesheet = mine.css\n")
(theme / "static").mkdir()
(theme / "static" / "mine.css").write_text("body { color: #111; }\n")
(src / "conf.py").write_text("project = 'M'\nhtml_theme = 'mine'\nhtml_theme_path = ['_themes']\n")
(src / "index.rst").write_text("M\n=\n\nOwn theme.\n")
assert build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")]) == 0
assert "mine.css" in (src / "_out" / "index.html").read_text()